In [1]:
# Numerical operations
import numpy as np

# Plotting
import matplotlib.pyplot as plt

# Interactive sliders
import ipywidgets as widgets
from ipywidgets import interact

plt.rcParams["figure.figsize"] = (8,5)
plt.rcParams["font.size"] = 11

In [2]:
# ---------------------------------------------------------
# 1. Angular velocity of LEO satellite
# ---------------------------------------------------------
def omega_leo(q_km, phi_deg):
    """
    q_km    : satellite height in km
    phi_deg : zenith angle in degrees
    
    Returns angular velocity in deg/sec
    """
    RE = 6378  # Earth radius in km
    q0 = q_km / RE
    
    # Eq (3): zenith angular velocity
    omega0 = 0.071 / (q0 * np.sqrt(1 + q0))
    
    # Eq (5): include zenith angle correction
    phi = np.deg2rad(phi_deg)
    omega = omega0 / (1/np.cos(phi))  # divide by sec(phi)
    
    return omega


# ---------------------------------------------------------
# 2. Pixel latency time (seconds)
# ---------------------------------------------------------
def pixel_latency(q_km, phi_deg, D, f, px_um):
    """
    D      : telescope diameter (m)
    f      : f-number
    px_um  : pixel size (micron)
    """
    omega = omega_leo(q_km, phi_deg)
    
    px_mm = px_um * 1e-3  # convert micron to mm
    
    # Eq (12) in seconds
    s_px_micro = (57 / omega) * (px_mm / (f * D))
    
    return s_px_micro * 1e-6  # convert microsecond to seconds


# ---------------------------------------------------------
# 3. Field of View (degrees)
# ---------------------------------------------------------
def fov(D, f, sensor_mm):
    """
    sensor_mm : detector size in mm
    """
    return 3.4 * (sensor_mm / 60) * (1 / (f * D))


# ---------------------------------------------------------
# 4. Magnitude limit model
# ---------------------------------------------------------
def mag_limit(D, s_exp):
    """
    Simplified photon-limited model
    """
    m0 = 10 + 2.5 * np.log10(D)  # reference scaling
    return m0 + 1.25 * np.log10(s_exp)


# ---------------------------------------------------------
# 5. Star density model
# ---------------------------------------------------------
def star_density(m_lim):
    """
    Bahcall-type star count approximation
    stars per square degree
    """
    return 10**(0.3*(m_lim - 12))


# ---------------------------------------------------------
# 6. Number of stars in FOV
# ---------------------------------------------------------
def star_count(D, f, sensor_mm, s_exp):
    FOV = fov(D, f, sensor_mm)
    m_lim = mag_limit(D, s_exp)
    
    return star_density(m_lim) * (FOV**2)

In [3]:
def exposure_analysis(q_km=500,
                      phi_deg=20,
                      D=0.2,
                      f=5,
                      px_um=3.76,
                      sensor_mm=22):
    
    # Exposure range (log scale)
    s_exp = np.logspace(-4, 1, 500)  # 0.0001s to 10s
    
    # Compute curves
    stars = [star_count(D, f, sensor_mm, s) for s in s_exp]
    mlim = [mag_limit(D, s) for s in s_exp]
    
    # Pixel latency
    s_px = pixel_latency(q_km, phi_deg, D, f, px_um)
    
    # Find minimum exposure giving 10 stars
    s_star = None
    for s in s_exp:
        if star_count(D, f, sensor_mm, s) >= 10:
            s_star = s
            break
    
    # Optimal exposure
    if s_star is not None:
        s_opt = min(s_px, s_star)
    else:
        s_opt = None
    
    # Plot
    plt.figure()
    plt.loglog(s_exp, stars, label="Number of Stars")
    plt.axhline(10, linestyle="--", label="Required 10 Stars")
    plt.axvline(s_px, linestyle="--", label="Pixel Latency Limit")
    
    if s_star:
        plt.axvline(s_star, linestyle=":", label="Star Requirement")
    
    if s_opt:
        plt.axvline(s_opt, linewidth=2, label="Optimal Exposure")
    
    plt.xlabel("Exposure Time (s)")
    plt.ylabel("Stars in FOV")
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # Print summary
    print("Pixel latency limit (s):", s_px)
    print("Exposure for 10 stars (s):", s_star)
    print("Optimal exposure (s):", s_opt)

In [ ]:
interact(exposure_analysis,
         q_km=widgets.FloatSlider(min=300, max=1200, step=50, value=500),
         phi_deg=widgets.FloatSlider(min=0, max=70, step=5, value=20),
         D=widgets.FloatSlider(min=0.1, max=0.5, step=0.05, value=0.2),
         f=widgets.FloatSlider(min=3, max=10, step=0.5, value=5),
         px_um=widgets.FloatSlider(min=2, max=6, step=0.2, value=3.76),
         sensor_mm=widgets.FloatSlider(min=10, max=40, step=2, value=22)
        );

interactive(children=(FloatSlider(value=500.0, description='q_km', max=1200.0, min=300.0, step=50.0), FloatSli…